# Pyridine-loaded Fourier-transform infrared spectroscopy (Py-IR)
Finely ground calcined samples were pressed in a hydraulic press (Perkin Elmer) at 125 bar (10 t) for 10 min and cut to shape to result in a rectangular wafer. The Nicolet IS10 IR spectrometer (Thermo Fischer Scientific) was loaded with the sample and heated to 100 °C for 1 h in high vacuum. Subsequently, the temperature was increased to 450 °C (10 K/min) and kept for 6 h to remove physiosorbed water. After cooling to 150 °C, a reference spectrum of the unloaded, dry sample was acquired. Subsequently, the sample was loaded with pyridine for 30 min at 4 mbar. Unabsorbed pyridine was removed again by applying high vacuum for 1 h. Measurement were carried out between 150 and 450 °C in 50 °C-steps with the temperature being kept constant for 1 h each. Per spectra 32 scans with a resolution of 4 cm $^{-1}$ were acquired.

1. Data correction

For quantification fo acid sites only the ring breathing vibrations in the range of 1400 cm $^{-1}$ < $\tilde{\nu}$ < 1600 cm $^{-1}$ are considered, as the characeteristic bands appear in this region. The data is truncated accordingly. Background correction is performed by subtracting the measurement without pyridine loading, the background file is stored in the JSON file for each measurement. A beaseline correction is done with the value closest to $\tilde{\nu}$ = 1525 cm $^{-1}$, the value of the baseline is stored in the JSON file for each measurement.

***
    1.1 Import required packages

In [ ]:
from pathlib import Path

from modules.data_processer import IRDataHandler
from modules.peak_fitter import PeakFitter, FitConfig


print("All done.")

---
    1.2 Select data
In your current working directory (`cwd`), specify the `folder` containing the data which are to be processed. The list of available files gives an overview over all samples in this specific folder.

In [ ]:
# Setup paths and configuration
folder = "OMAS_1-5_P123-Cy_T80"
cwd = Path.cwd()
path_to_directory = cwd / folder

# Initialize data handler
data_handling = IRDataHandler(path_to_directory=path_to_directory, folder = folder, decimal=",")
data_handling.available_files()

---
    1.3 Process data
Plot the IR curves for visualization and save the corrected data to `folder / corr` for the analysis. Save all sample metadata to a new JSON file (for changing existing metadata go to `2.2`).

In [ ]:
# Process and plot data
data_handling.get_plot()

In [ ]:
# Save metadata
data_handling.add_sample_metadata(
    sample_mass = 0.1002,
    sample_length = 1.832,
    sample_width= 0.928,
    extinction_coefficient_bronsted=1.67,
    extinction_coefficient_lewis=2.22,
    error_sample_length=0.001,
    error_sample_width=0.001,
    error_sample_mass=0.0001,
    surface_area=348.0, # OPTIONAL: surface area in m²/g
    error_surface_area=None,# OPTIONAL: error in surface area
)
data_handling.save_json(json_filename=f"{folder}.json")

***

2. Fitting

According to the Beer–Lambert law, the logarithm of the intensity ratio is proportional to the product of the extinction coefficient (ε), concentration (c), and optical path length (d). For solid-state measurements, the path length is not directly accessible. Therefore, we replaced this term with the area density (mass per unit area), allowing for a practical expression of the acid site density. Specifically, the number of acid sites was calculated using the relation:

$$N = \frac{A_{sample} \cdot A_{peak}}{m_{sample} \cdot \varepsilon}$$

where $A_{sample}$ is the exposed sample area, $A_{peak}$ the integrated peak area (obtained over the relevant wavenumber range), ε the extinction coefficient, and $m_{sample}$ the sample mass. The extinction coefficient was taken from literature values reported for aluminosilicate materials (C.A. Emeis, ), providing reasonable approximation given the chemical similarity. Despite the lack of internal standards and absolute calibration, this method enables quantitative comparison, when all measurements are performed on the same instrument under identical experimental conditions. Error propagation analysis is possible, if uncertainties of the values are given in the sample metadata (uncertainty of the mass, length, width). A Voigt fit is performed to account for both homegeneous (Lorentz) and heterogeneous (Gauss) peak broadening.

***
    2.1 initialize the PeakFitter class
specifiy a `threshold` if necessary and check the available files.

In [ ]:
# Setup peak fitiing
corr = Path(folder) / "corr"

# Create fit configuration
fit_config = FitConfig(
    threshold=0.01
)

# Initialize PeakFitter
peak_fitter = PeakFitter(
    path_to_directory= cwd / corr,
    folder=corr,
    fit_config=fit_config
)
# Print available files
peak_fitter.available_files()


---
    2.2 Correct sample metadata
This cell corrects all calculated values in you JSON with the new sample metadata. Run only if needed!

In [ ]:
# correct JSON with new sample metadata
path_to_json = cwd / folder / "fits_plots"

peak_fitter.recalculate_acid_sites(
    sample_mass = 0.0840,
    sample_length = 1.970,
    sample_width= 0.970,
    extinction_coefficient_bronsted=1.67,
    extinction_coefficient_lewis=2.22,
    error_sample_length=0.001,
    error_sample_width=0.001,
    error_sample_mass=0.0001,
    surface_area=351.0, # OPTIONAL: surface area in m²/g
    error_surface_area=None,# OPTIONAL: error in surface area
    json_filename=path_to_json / f"{folder}.json"
)

---
    2.3 Get control plot
Get the control plot for the smaple indicated by `index` and check if all peaks are found and if a shoulder is visible but not detected.

In [ ]:
# Add sample meta data
path_to_json = cwd / folder / "fits_plots"
print(path_to_json)
peak_fitter.load_metadata_from_json(path_to_json / f"{folder}.json")

# Extract data and find peaks
index = 6

# Create output filename
name = "voigt_fit"
print(peak_fitter.available_files()[index])

peak_fitter.extract_data(index=index)
peak_fitter.get_control_plot(index=index)

    2.4 Perform fitting
Indicate if a shoulder fit should be performed (`shoulder`), and if so, for which peak (lewis, mixed or bronsted). Then run the fit and check the results. The fitting uses the parameters from the `peak_finder` as initial guesses and improves over a maximum of 20 attempts per peak. If the result is not necessary, try running it again.

In [ ]:
# does the Lewis peak have a shoulder?
shoulder = False #Boolean (True/False)
peak_fitter.force_shoulder("lewis", force=shoulder)

#Run the main analysis
peak_fitter.analyze()

# View results (area, shoulder, detection)
#peak_fitter.print_results()
save_path = cwd / data_handling.folder
peak_fitter.plot_results(index=index, name=name, save_path=save_path)

    2.5 Save data to JSON
Finally, with the data obtained from the fits, calculate the number of acid sites and save everything to the existing JSON.

In [ ]:
# Calculate number of acid sites
n_sites = peak_fitter.calc_n_sites()
print("\nCalculated number of acid sites (mmol/g):")
print(f"Bronsted sites: {n_sites[0]:.3f} +/- {n_sites[1]} \u03BCmol/g")
print(f"Lewis sites: {n_sites[2]:.3f} +/- {n_sites[3]} \u03BCmol/g")

# save results to existing json
peak_fitter.update_json(json_filename=path_to_json / f"{data_handling.folder}.json", index=index)